Testing

In [1]:
# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import ast
import pandas as pd
import difflib

# 2. 定義基礎路徑
BASE_DIR = '/content/drive/MyDrive/SMC2024/'
EXCEL_PATH = os.path.join(BASE_DIR, 'SMC2024.xlsx')
EXP_ID_PATH = os.path.join(BASE_DIR, 'experiment ID.xls')
OUTPUT_EXCEL_PATH = os.path.join(BASE_DIR, 'SMC2024_auto_filled_testing.xlsx')

# 限制處理範圍
TARGET_FOLDERS = [f"1_{i}" for i in range(1, 14)]
VALID_FILENAMES = ['inputmsglog.txt', 'inputinterpretation.txt', 'outputinterpretation.txt']

# 3. 定義 GT 資訊
GT_INFO = {
    r"gt/SMC_gounod_16_1/outputscore.txt": ("gounod ave maria accompany_前半", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_16_1/outputscore_concat.txt": ("gounod ave maria accompany_Concatenation", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_16_2/outputscore.txt": ("gounod ave maria accompany_後半", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_melody_1/outputscore.txt": ("gounod ave maria melody_前半", "gounod ave maria", "RH"),
    r"gt/SMC_gounod_melody_1/outputscore_concat.txt": ("gounod ave maria melody_Concatenation", "gounod ave maria", "RH"),
    r"gt/SMC_gounod_melody_2/outputscore.txt": ("gounod ave maria melody_後半", "gounod ave maria", "RH"),
    r"gt/SMC_little_star_LH4/outputscore.txt": ("little star theme", "little star theme", "LH"),
    r"gt/SMC_little_star_LH16/outputscore.txt": ("little star var2", "little star var2", "LH"),
    r"gt/SMC_little_star_RH/outputscore.txt": ("little star for both theme and var2", "little star", "RH")
}

# 4. 輔助功能
def extract_pitch_sequence(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = ast.literal_eval(f.read().strip())
            return [row[2] for row in data if len(row) >= 3 and row[1] == 144]
    except: return []

# 預載 GT 序列
gt_sequences = {}
for rel_path, info in GT_INFO.items():
    sys_path = os.path.join(BASE_DIR, rel_path.replace('\\', '/'))
    gt_sequences[rel_path] = {"full": info[0], "base": info[1], "hand": info[2], "seq": extract_pitch_sequence(sys_path)}

# 載入 Exp ID 對照表
exp_df = pd.read_excel(EXP_ID_PATH)
exp_dict = pd.Series(exp_df['person'].values, index=exp_df['ID'].astype(str)).to_dict()

# 5. 讀取原始 Excel
print("正在讀取原始清單並準備遍歷資料夾...")
main_df = pd.read_excel(EXCEL_PATH)
num_columns = len(main_df.columns) # 動態獲取實際的欄位數量 (你原本的是 12)

# 取得目前已有的路徑集合，避免重複填寫
existing_paths = set(main_df.iloc[:, 1].dropna().astype(str).str.strip(' "').str.replace('\\', '/'))
# 取得最後一個 Entry 編號
if not main_df.empty:
    last_entry = main_df.iloc[:, 0].max()
    if pd.isna(last_entry): last_entry = 0
else:
    last_entry = 0

new_rows = []

# 6. 主動遍歷 1_1 到 1_13 資料夾
for folder_id in TARGET_FOLDERS:
    folder_path = os.path.join(BASE_DIR, 'data', folder_id)
    if not os.path.exists(folder_path):
        continue

    for fname in os.listdir(folder_path):
        if fname not in VALID_FILENAMES:
            continue

        # 建立相對路徑格式
        rel_path = f"data\\{folder_id}\\{fname}"
        clean_rel_path = rel_path.replace('\\', '/')

        # 如果路徑已經在 Excel 裡面了，就跳過
        if clean_rel_path in existing_paths:
            continue

        # --- 開始產生新資料列 ---
        last_entry += 1
        entry_id = last_entry
        sys_file_path = os.path.join(folder_path, fname)

        # 初始化各欄位
        score_val = "?"
        player_val = "?"
        time_val = "?"
        type_val = ""
        code_val = "X"
        notes_val = ""
        note_prefix = ""

        # 判定 Player 與 Type
        if 'input' in fname.lower():
            if folder_id in exp_dict: player_val = str(exp_dict[folder_id])
            else:
                player_val = "??"
                note_prefix = f"[找不到 ID: {folder_id}] "
            type_val = 'inputmslog' if 'msglog' in fname else 'inputintepretation'
        else:
            indicator_files = ['receivednotelog.txt', 'realoutputlog.txt', 'outputtiminglog.txt']
            is_ai = any(os.path.exists(os.path.join(folder_path, f)) for f in indicator_files)
            player_val = "AI" if is_ai else "kit?"
            type_val = 'outputintepretation'

        if 'interpretation' in fname.lower(): code_val = 'extract_interpretation.py'

        # 比對曲目 (Score) 與 左右手
        test_seq = extract_pitch_sequence(sys_file_path)
        if test_seq:
            best_match, max_acc = None, -1
            for k, gt in gt_sequences.items():
                if not gt['seq']: continue
                matcher = difflib.SequenceMatcher(None, test_seq, gt['seq'])
                acc = sum(n for i,j,n in matcher.get_matching_blocks()) / len(gt['seq'])
                if acc > max_acc: max_acc, best_match = acc, gt

            if best_match:
                acc_p = round(max_acc * 100, 1)
                if 0.40 <= max_acc <= 0.65:
                    score_val, hand_val = best_match['base'], "Both"
                else:
                    score_val, hand_val = best_match['full'], best_match['hand']
                notes_val = f"{note_prefix}{hand_val}, Acc: {acc_p}%"

        # 【重要修正】動態產生符合 Excel 總欄位長度的空白 list
        new_row = [""] * num_columns

        # 填入我們確實知道的 9 個欄位資料
        new_row[0] = entry_id
        new_row[1] = rel_path
        new_row[2] = score_val
        new_row[3] = player_val
        new_row[4] = time_val
        new_row[5] = type_val
        new_row[6] = code_val
        new_row[7] = "" # Concurrent recordings 暫不填
        new_row[8] = notes_val
        # 如果有第 10, 11, 12 個欄位，它們會保持原本預設的 ""

        new_rows.append(new_row)
        print(f"新增數據: {folder_id}/{fname} ({notes_val})")

# 7. 合併並儲存
if new_rows:
    new_df = pd.DataFrame(new_rows, columns=main_df.columns)
    final_df = pd.concat([main_df, new_df], ignore_index=True)
    final_df.to_excel(OUTPUT_EXCEL_PATH, index=False)
    print(f"\n自動填寫完成！已從 Entry {new_rows[0][0]} 新增至 {last_entry}。")
    print(f"檔案已儲存至: {OUTPUT_EXCEL_PATH}")
else:
    print("\n沒有發現新的檔案需要填寫。")

Mounted at /content/drive
正在讀取原始清單並準備遍歷資料夾...
新增數據: 1_8/inputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_8/inputmsglog.txt (RH, Acc: 86.2%)
新增數據: 1_8/outputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_10/inputmsglog.txt (RH, Acc: 70.5%)
新增數據: 1_10/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_10/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_11/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_11/inputmsglog.txt (LH, Acc: 100.0%)
新增數據: 1_11/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_12/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_12/inputmsglog.txt (LH, Acc: 100.0%)
新增數據: 1_12/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_13/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_13/inputmsglog.txt (LH, Acc: 97.1%)
新增數據: 1_13/inputinterpretation.txt (LH, Acc: 100.0%)

自動填寫完成！已從 Entry 41 新增至 55。
檔案已儲存至: /content/drive/MyDrive/SMC2024/SMC2024_auto_filled_testing.xlsx


正式跑全部

In [2]:
# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import ast
import pandas as pd
import difflib
import re  # <--- 新增：用來辨識資料夾名稱格式

# 2. 定義基礎路徑
BASE_DIR = '/content/drive/MyDrive/SMC2024/'
DATA_DIR = os.path.join(BASE_DIR, 'data')
EXCEL_PATH = os.path.join(BASE_DIR, 'SMC2024.xlsx')
EXP_ID_PATH = os.path.join(BASE_DIR, 'experiment ID.xls') # 注意：副檔名請確認是 xls 還是 xlsx
OUTPUT_EXCEL_PATH = os.path.join(BASE_DIR, 'SMC2024_auto_filled_testing2.xlsx') # 改名為 FINAL

# ---------------------------------------------------------
# 【關鍵修改點】：自動掃描 data 資料夾下所有的 X_Y 資料夾
# ---------------------------------------------------------
print("正在掃描 data 資料夾...")
TARGET_FOLDERS = []
if os.path.exists(DATA_DIR):
    for f in os.listdir(DATA_DIR):
        # 條件：1. 必須是資料夾 2. 名稱格式必須是 "數字_數字" (例如 1_1, 18_6)
        if os.path.isdir(os.path.join(DATA_DIR, f)) and re.match(r'^\d+_\d+$', f):
            TARGET_FOLDERS.append(f)

    # 按照數字大小進行自然排序 (讓 1_2 排在 1_10 前面，而不是字串排序的 1_10 -> 1_2)
    TARGET_FOLDERS.sort(key=lambda x: [int(p) for p in x.split('_')])
    print(f"共找到 {len(TARGET_FOLDERS)} 個實驗資料夾準備處理！")
else:
    print(f"找不到資料夾: {DATA_DIR}")

VALID_FILENAMES = ['inputmsglog.txt', 'inputinterpretation.txt', 'outputinterpretation.txt']

# 3. 定義 GT 資訊
GT_INFO = {
    r"gt/SMC_gounod_16_1/outputscore.txt": ("gounod ave maria accompany_前半", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_16_1/outputscore_concat.txt": ("gounod ave maria accompany_Concatenation", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_16_2/outputscore.txt": ("gounod ave maria accompany_後半", "gounod ave maria", "LH"),
    r"gt/SMC_gounod_melody_1/outputscore.txt": ("gounod ave maria melody_前半", "gounod ave maria", "RH"),
    r"gt/SMC_gounod_melody_1/outputscore_concat.txt": ("gounod ave maria melody_Concatenation", "gounod ave maria", "RH"),
    r"gt/SMC_gounod_melody_2/outputscore.txt": ("gounod ave maria melody_後半", "gounod ave maria", "RH"),
    r"gt/SMC_little_star_LH4/outputscore.txt": ("little star theme", "little star theme", "LH"),
    r"gt/SMC_little_star_LH16/outputscore.txt": ("little star var2", "little star var2", "LH"),
    r"gt/SMC_little_star_RH/outputscore.txt": ("little star for both theme and var2", "little star", "RH")
}

# 4. 輔助功能
def extract_pitch_sequence(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = ast.literal_eval(f.read().strip())
            return [row[2] for row in data if len(row) >= 3 and row[1] == 144]
    except: return []

# 預載 GT 序列
gt_sequences = {}
for rel_path, info in GT_INFO.items():
    sys_path = os.path.join(BASE_DIR, rel_path.replace('\\', '/'))
    gt_sequences[rel_path] = {"full": info[0], "base": info[1], "hand": info[2], "seq": extract_pitch_sequence(sys_path)}

# 載入 Exp ID 對照表
exp_df = pd.read_excel(EXP_ID_PATH)
exp_dict = pd.Series(exp_df['person'].values, index=exp_df['ID'].astype(str)).to_dict()

# 5. 讀取原始 Excel
print("正在讀取原始清單並準備開始比對...")
main_df = pd.read_excel(EXCEL_PATH)
num_columns = len(main_df.columns)

# 取得目前已有的路徑集合，避免重複填寫
existing_paths = set(main_df.iloc[:, 1].dropna().astype(str).str.strip(' "').str.replace('\\', '/'))

if not main_df.empty:
    last_entry = main_df.iloc[:, 0].max()
    if pd.isna(last_entry): last_entry = 0
else:
    last_entry = 0

new_rows = []

# 6. 主動遍歷所有抓到的資料夾
for folder_id in TARGET_FOLDERS:
    folder_path = os.path.join(BASE_DIR, 'data', folder_id)

    for fname in os.listdir(folder_path):
        if fname not in VALID_FILENAMES:
            continue

        rel_path = f"data\\{folder_id}\\{fname}"
        clean_rel_path = rel_path.replace('\\', '/')

        if clean_rel_path in existing_paths:
            continue

        last_entry += 1
        entry_id = last_entry
        sys_file_path = os.path.join(folder_path, fname)

        score_val = "?"
        player_val = "?"
        time_val = "?"
        type_val = ""
        code_val = "X"
        notes_val = ""
        note_prefix = ""

        if 'input' in fname.lower():
            if folder_id in exp_dict: player_val = str(exp_dict[folder_id])
            else:
                player_val = "??"
                note_prefix = f"[找不到 ID: {folder_id}] "
            type_val = 'inputmslog' if 'msglog' in fname else 'inputintepretation'
        else:
            indicator_files = ['receivednotelog.txt', 'realoutputlog.txt', 'outputtiminglog.txt']
            is_ai = any(os.path.exists(os.path.join(folder_path, f)) for f in indicator_files)
            player_val = "AI" if is_ai else "kit?"
            type_val = 'outputintepretation'

        if 'interpretation' in fname.lower(): code_val = 'extract_interpretation.py'

        test_seq = extract_pitch_sequence(sys_file_path)
        if test_seq:
            best_match, max_acc = None, -1
            for k, gt in gt_sequences.items():
                if not gt['seq']: continue
                matcher = difflib.SequenceMatcher(None, test_seq, gt['seq'])
                acc = sum(n for i,j,n in matcher.get_matching_blocks()) / len(gt['seq'])
                if acc > max_acc: max_acc, best_match = acc, gt

            if best_match:
                acc_p = round(max_acc * 100, 1)
                if 0.40 <= max_acc <= 0.65:
                    score_val, hand_val = best_match['base'], "Both"
                else:
                    score_val, hand_val = best_match['full'], best_match['hand']
                notes_val = f"{note_prefix}{hand_val}, Acc: {acc_p}%"
        else:
            notes_val = f"{note_prefix}[檔案無音符數據]"

        new_row = [""] * num_columns
        new_row[0] = entry_id
        new_row[1] = rel_path
        new_row[2] = score_val
        new_row[3] = player_val
        new_row[4] = time_val
        new_row[5] = type_val
        new_row[6] = code_val
        new_row[7] = ""
        new_row[8] = notes_val

        new_rows.append(new_row)
        print(f"新增數據: {folder_id}/{fname} ({notes_val})")

# 7. 合併並儲存
if new_rows:
    new_df = pd.DataFrame(new_rows, columns=main_df.columns)
    final_df = pd.concat([main_df, new_df], ignore_index=True)
    final_df.to_excel(OUTPUT_EXCEL_PATH, index=False)
    print(f"\n======================================")
    print(f"自動填寫完成！共新增了 {len(new_rows)} 筆資料。")
    print(f"已從 Entry {new_rows[0][0]} 填寫至 {last_entry}。")
    print(f"檔案已儲存至: {OUTPUT_EXCEL_PATH}")
    print(f"======================================")
else:
    print("\n沒有發現新的檔案需要填寫。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在掃描 data 資料夾...
共找到 123 個實驗資料夾準備處理！
正在讀取原始清單並準備開始比對...
新增數據: 1_8/inputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_8/inputmsglog.txt (RH, Acc: 86.2%)
新增數據: 1_8/outputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_10/inputmsglog.txt (RH, Acc: 70.5%)
新增數據: 1_10/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_10/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_11/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_11/inputmsglog.txt (LH, Acc: 100.0%)
新增數據: 1_11/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_12/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_12/inputmsglog.txt (LH, Acc: 100.0%)
新增數據: 1_12/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 1_13/outputinterpretation.txt (RH, Acc: 100.0%)
新增數據: 1_13/inputmsglog.txt (LH, Acc: 97.1%)
新增數據: 1_13/inputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 2_1/outputinterpretation.txt (LH, Acc: 100.0%)
新增數據: 2_